<a href="https://colab.research.google.com/github/AustineLomocso/PocketClass_Backend/blob/master/pocky_fixed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Grade 5 Science MCQ Fine-Tuning Pipeline
**Model:** Phi-3-mini-4k-instruct (4-bit) + LoRA → GGUF Q4_K_M

**Fixes applied:**
- `import re` added (was missing, caused NameError)
- `chunk_by_lesson_section()` is now called and assigned to `chunks`
- `extract_json()` helper strips markdown fences before saving training data
- GGUF path corrected in inference cell
- Testing cells added (in-notebook, batch validation, GGUF)


In [2]:
# ── CELL 1: Install dependencies ────────────────────────────────
!pip install unsloth pymupdf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.6/62.6 MB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 86.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 33.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 100.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 42.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 418.4/418.4 kB 47.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 124.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 135.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [6]:
# ── CELL 2: PDF extraction + chunking + MCQ self-distillation ───

import fitz          # PyMuPDF
import json
import re            # FIX 1: was missing; chunk_by_lesson_section uses re.split / re.match
import torch
from unsloth import FastLanguageModel


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# STAGE 1 — Extract text from your PDF
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# How it works: fitz opens each page and pulls out raw text.
# We join it all, then collapse multiple spaces/newlines into
# single spaces so the text is clean for chunking.

def extract_pdf(pdf_path: str) -> str:
    doc = fitz.open(pdf_path)
    pages = [page.get_text() for page in doc]
    full_text = " ".join(pages)
    return " ".join(full_text.split())   # normalize whitespace

raw_text = extract_pdf("/content/GRADE 5 SCIENCE.pdf")   # <-- upload your PDF first
print(f"Extracted {len(raw_text)} characters from PDF")


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# STAGE 2 — Chunk the text into lesson-sized passages
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# How it works: we split on the PDF's natural section headers so
# each chunk is one coherent unit — a definitions block,
# explanation block, or examples block. This keeps each chunk
# semantically coherent rather than cutting mid-sentence the
# way a fixed character split would.

def chunk_by_lesson_section(text: str) -> list:
    """
    Split on the PDF's natural section breaks instead of arbitrary
    character counts. Each chunk is one coherent unit: a definitions
    block, explanation block, or examples block.
    """
    section_markers = [
        "Definitions of Terms:",
        "Deep Explanation:",
        "Examples:",
        "Practice Quiz:"
    ]

    chunks = []
    # First split into per-lesson blocks using the Lesson N: pattern
    lesson_blocks = re.split(r'(Lesson \d+:)', text)

    current_lesson_title = ""
    for i, block in enumerate(lesson_blocks):
        if re.match(r'Lesson \d+:', block):
            current_lesson_title = block.strip()
            continue
        if not block.strip() or len(block.strip()) < 100:
            continue

        # Now split this lesson block by section markers
        sub_sections = re.split(
            r'(Definitions of Terms:|Deep Explanation:|Examples:|Practice Quiz:)',
            block
        )

        current_section = ""
        current_header = ""
        for part in sub_sections:
            if part in section_markers:
                if len(current_section.strip()) > 80:
                    full_chunk = f"{current_lesson_title} {current_header} {current_section}".strip()
                    chunks.append(full_chunk)
                current_header = part
                current_section = ""
            else:
                current_section += part

        # Don't forget the last section in the block
        if len(current_section.strip()) > 80:
            full_chunk = f"{current_lesson_title} {current_header} {current_section}".strip()
            chunks.append(full_chunk)

    # Filter out "Practice Quiz:" chunks — those have the answers already
    content_chunks = [c for c in chunks if "Practice Quiz:" not in c]
    return content_chunks


# FIX 2: actually call the function and assign result to `chunks`
# (original code defined the function but never called it,
#  causing NameError when Stage 3 iterated over `chunks`)
chunks = chunk_by_lesson_section(raw_text)
print(f"Split into {len(chunks)} chunks — preview of first chunk:")
print(chunks[0])
print()


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# HELPER — Strip markdown fences before saving training data
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# FIX 3: Phi-3 Mini often wraps JSON output in ```json ... ```
# fences or adds a prose preamble. Saving that raw text makes
# the training targets inconsistent and breaks json.loads() in
# your mobile app. This helper normalises every MCQ to bare JSON.

def extract_json(text: str) -> str:
    """Strip markdown fences and extract the first JSON object."""
    # Remove ```json ... ``` or ``` ... ``` fences
    text = re.sub(r"```(?:json)?", "", text).strip()
    # Find the first complete { ... } block
    match = re.search(r"\{.*\}", text, re.DOTALL)
    return match.group(0).strip() if match else text.strip()


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# STAGE 3 — Use the base model to generate MCQ answers
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# How it works: we load Phi-3 in INFERENCE mode (no LoRA yet)
# and ask it to produce an MCQ for every chunk. This is called
# "self-distillation" — the base model acts as a teacher to
# label the data. We save each (chunk → MCQ) pair as a
# training example. Fine-tuning then teaches the model to
# produce consistently structured MCQs specifically for YOUR
# PDF's content and vocabulary.

print("\nLoading Phi-3 Mini for MCQ generation...")
gen_model, gen_tokenizer = FastLanguageModel.from_pretrained(
    model_name     = "unsloth/Phi-3-mini-4k-instruct-bnb-4bit",
    max_seq_length = 1024,
    load_in_4bit   = True,
)
FastLanguageModel.for_inference(gen_model)  # enables fast inference mode

SYSTEM_PROMPT = (
    "You are a Grade 5 Science teacher. "
    "When given a passage, generate one multiple choice question. "
    "Output ONLY valid JSON with exactly these keys: "
    "question, options (object with keys A B C D), answer_key, rationale. "
    "Do NOT include any markdown, code fences, or explanatory text outside the JSON."
)

def generate_mcq_from_chunk(chunk: str) -> str:
    messages = [
        {"role": "user",
         "content": f"{SYSTEM_PROMPT}\n\nPassage:\n{chunk}"}
    ]
    input_ids = gen_tokenizer.apply_chat_template(
        messages,
        return_tensors        = "pt",
        add_generation_prompt = True
    ).to("cuda")

    output = gen_model.generate(
        input_ids,
        max_new_tokens = 350,
        temperature    = 0.7,
        do_sample      = True,
    )
    # Slice off the prompt tokens so we only decode the new MCQ
    response = gen_tokenizer.decode(
        output[0][input_ids.shape[1]:],
        skip_special_tokens = True
    )
    return response.strip()


# Generate one training example per chunk
raw_data = []
skipped  = 0
for i, chunk in enumerate(chunks):
    print(f"  [{i+1}/{len(chunks)}] Generating MCQ...")
    mcq_raw  = generate_mcq_from_chunk(chunk)
    mcq_clean = extract_json(mcq_raw)   # FIX 3: normalise before saving

    # Light validation: skip entries that couldn't produce JSON at all
    try:
        parsed = json.loads(mcq_clean)
        if not all(k in parsed for k in ["question", "options", "answer_key"]):
            raise ValueError("Missing required keys")
    except Exception as e:
        print(f"    ⚠️  Skipping chunk {i+1} — bad JSON: {e}")
        skipped += 1
        continue

    raw_data.append({
        "messages": [
            {
                "role": "user",
                "content": f"{SYSTEM_PROMPT}\n\nPassage:\n{chunk}"
            },
            {
                "role": "assistant",   # ← "assistant" for Phi-3, NOT "model"
                "content": mcq_clean  # clean JSON, no fences
            }
        ]
    })

# Save to JSONL
with open("science_train.jsonl", "w") as f:
    for entry in raw_data:
        f.write(json.dumps(entry) + "\n")

# Free the generation model's VRAM before training starts
del gen_model
torch.cuda.empty_cache()

print(f"\nSaved {len(raw_data)} training examples to science_train.jsonl")
print(f"Skipped {skipped} chunks due to malformed JSON output")

Extracted 38047 characters from PDF
Split into 97 chunks — preview of first chunk:
GRADE 5 SCIENCE: COMPLETE MATATAG CURRICULUM LESSON MATERIALS QUARTER 1: MATERIALS (Matter and Scientific Investigation)


Loading Phi-3 Mini for MCQ generation...
==((====))==  Unsloth 2026.4.4: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.26G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/194 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

  [95/97] Generating MCQ...


Both `max_new_tokens` (=350) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


    ⚠️  Skipping chunk 95 — bad JSON: Expecting ',' delimiter: line 15 column 4 (char 353)
  [96/97] Generating MCQ...


Both `max_new_tokens` (=350) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [97/97] Generating MCQ...
    ⚠️  Skipping chunk 97 — bad JSON: Expecting property name enclosed in double quotes: line 10 column 1 (char 208)

Saved 66 training examples to science_train.jsonl
Skipped 31 chunks due to malformed JSON output


In [1]:
# ── CELL 3: LoRA fine-tuning + GGUF export ──────────────────────

from unsloth import FastLanguageModel
import torch
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments

# ── 1. Load Phi-3 Mini (fits in T4's 15GB with room to train) ──
max_seq_length = 1024
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = "unsloth/Phi-3-mini-4k-instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype          = None,
    load_in_4bit   = True,
)

# ── 2. LoRA adapter ─────────────────────────────────────────────
model = FastLanguageModel.get_peft_model(
    model,
    r              = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha     = 32,
    lora_dropout   = 0,
    bias           = "none",
    use_gradient_checkpointing = "unsloth",
    random_state   = 3407,
)

# ── 3. Load your PDF-generated dataset ──────────────────────────
dataset = load_dataset("json", data_files="science_train.jsonl", split="train")

def format_chat_template(examples):
    texts = [
        tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False)
        for convo in examples["messages"]
    ]
    return {"text": texts}

dataset = dataset.map(format_chat_template, batched=True)

# ── 4. Train ─────────────────────────────────────────────────────
trainer = SFTTrainer(
    model              = model,
    tokenizer          = tokenizer,
    train_dataset      = dataset,
    dataset_text_field = "text",
    max_seq_length     = max_seq_length,
    dataset_num_proc   = 2,
    packing            = False,
    args = TrainingArguments(
        per_device_train_batch_size  = 2,
        gradient_accumulation_steps  = 4,
        warmup_steps                 = 2,
        num_train_epochs             = 12,
        learning_rate                = 2e-4,
        fp16                         = True,
        bf16                         = False,
        logging_steps                = 10,
        optim                        = "adamw_8bit",
        weight_decay                 = 0.01,
        lr_scheduler_type            = "cosine",
        seed                         = 3407,
        output_dir                   = "outputs",
    ),
)

trainer_stats = trainer.train()
print(f"\nFinal training loss: {trainer_stats.training_loss:.4f}")

# ── 5. Save LoRA adapter ─────────────────────────────────────────
model.save_pretrained("lora_adapter")
tokenizer.save_pretrained("lora_adapter")
print("LoRA adapter saved to ./lora_adapter")

# ── 6. Export to GGUF for React Native app ───────────────────────
model.save_pretrained_gguf("phi3_science_model", tokenizer, quantization_method="q4_k_m")
print("Done — GGUF file is at phi3_science_model_gguf/phi-3-mini-4k-instruct.Q4_K_M.gguf")

ModuleNotFoundError: No module named 'unsloth'

In [ ]:
# ── CELL 4: IN-NOTEBOOK INFERENCE TEST (Unsloth, pre-GGUF) ──────
# Run this immediately after trainer.train() to verify the fine-tuned
# model produces valid, structured MCQs before you export to GGUF.

import json, re

def extract_json(text: str) -> str:
    """Strip markdown fences and return the first JSON object found."""
    text = re.sub(r"```(?:json)?", "", text).strip()
    match = re.search(r"\{.*\}", text, re.DOTALL)
    return match.group(0).strip() if match else text.strip()

FastLanguageModel.for_inference(model)

# Use the first chunk as a smoke-test passage
TEST_PASSAGE = chunks[0]

messages = [
    {"role": "user", "content": f"{SYSTEM_PROMPT}\n\nPassage:\n{TEST_PASSAGE}"}
]

input_ids = tokenizer.apply_chat_template(
    messages,
    return_tensors        = "pt",
    add_generation_prompt = True
).to("cuda")

output = model.generate(
    input_ids,
    max_new_tokens = 400,
    temperature    = 0.3,   # lower temp → more deterministic JSON structure
    do_sample      = True,
)

response = tokenizer.decode(
    output[0][input_ids.shape[1]:],
    skip_special_tokens = True
)

print("=" * 60)
print("RAW MODEL OUTPUT:")
print("=" * 60)
print(response)
print()

# ── Validate JSON structure ──────────────────────────────────────
REQUIRED_KEYS = {"question", "options", "answer_key", "rationale"}

print("=" * 60)
print("VALIDATION:")
print("=" * 60)
try:
    parsed = json.loads(extract_json(response))
    missing = REQUIRED_KEYS - parsed.keys()
    if missing:
        print(f"⚠️  Missing keys: {missing}")
    else:
        print("✅  Valid MCQ JSON — all required keys present")
        print(f"\nQuestion   : {parsed['question']}")
        for letter, text in parsed['options'].items():
            marker = " ◀ CORRECT" if letter == parsed['answer_key'] else ""
            print(f"  {letter}) {text}{marker}")
        print(f"\nRationale  : {parsed['rationale']}")
except json.JSONDecodeError as e:
    print(f"❌  JSON parse error: {e}")
    print("Cleaned text attempted:", extract_json(response)[:300])

In [ ]:
# ── CELL 5: BATCH QUALITY CHECK across all chunks ────────────────
# Runs inference on every chunk and reports JSON validity rate.
# Target: ≥85% valid. If lower, increase num_train_epochs to 10-12
# and lower temperature to 0.2 during inference.

import json, re
from tqdm import tqdm   # pip install tqdm — already available in Colab

def extract_json(text: str) -> str:
    text = re.sub(r"```(?:json)?", "", text).strip()
    match = re.search(r"\{.*\}", text, re.DOTALL)
    return match.group(0).strip() if match else text.strip()

REQUIRED_KEYS = {"question", "options", "answer_key", "rationale"}
results = []

for i, chunk in enumerate(tqdm(chunks, desc="Validating chunks")):
    msgs = [{"role": "user",
             "content": f"{SYSTEM_PROMPT}\n\nPassage:\n{chunk}"}]
    ids = tokenizer.apply_chat_template(
        msgs, return_tensors="pt", add_generation_prompt=True
    ).to("cuda")

    out = model.generate(
        ids,
        max_new_tokens = 400,
        temperature    = 0.3,
        do_sample      = True,
    )
    resp = tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True)

    status = "PASS"
    note   = ""
    try:
        parsed = json.loads(extract_json(resp))
        missing = REQUIRED_KEYS - parsed.keys()
        if missing:
            status = "FAIL"
            note   = f"Missing keys: {missing}"
        # Extra check: options must have A, B, C, D
        elif not all(k in parsed.get("options", {}) for k in ["A","B","C","D"]):
            status = "WARN"
            note   = "options dict missing A/B/C/D keys"
        # Extra check: answer_key must be one of the options
        elif parsed["answer_key"] not in parsed.get("options", {}):
            status = "WARN"
            note   = f"answer_key '{parsed['answer_key']}' not in options"
    except json.JSONDecodeError as e:
        status = "FAIL"
        note   = f"JSON error: {e} | Raw: {resp[:80]}"

    results.append({"chunk": i+1, "status": status, "note": note})

# ── Summary ───────────────────────────────────────────────────────
total  = len(results)
passed = sum(1 for r in results if r["status"] == "PASS")
warned = sum(1 for r in results if r["status"] == "WARN")
failed = sum(1 for r in results if r["status"] == "FAIL")

print(f"\n{'='*50}")
print(f"BATCH VALIDATION RESULTS — {total} chunks")
print(f"{'='*50}")
print(f"  ✅ PASS : {passed:3d}  ({100*passed/total:.1f}%)")
print(f"  ⚠️  WARN : {warned:3d}  ({100*warned/total:.1f}%)")
print(f"  ❌ FAIL : {failed:3d}  ({100*failed/total:.1f}%)")
print()

if failed > 0 or warned > 0:
    print("Issues detail:")
    for r in results:
        if r["status"] != "PASS":
            print(f"  Chunk {r['chunk']:3d} [{r['status']}] {r['note']}")

# Recommendation
pass_rate = passed / total
if pass_rate >= 0.90:
    print("\n🟢 Model quality is GOOD (≥90% pass). Safe to deploy.")
elif pass_rate >= 0.75:
    print("\n🟡 Model quality is ACCEPTABLE (75-90%). Consider adding epochs.")
else:
    print("\n🔴 Model quality is LOW (<75%). Increase num_train_epochs to 10-12 and retrain.")

In [ ]:
# ── TARGETED MCQ: Solid State of Matter ─────────────────────────
import json, re

def extract_json(text: str) -> str:
    text = re.sub(r"```(?:json)?", "", text).strip()
    match = re.search(r"\{.*\}", text, re.DOTALL)
    return match.group(0).strip() if match else text.strip()

SOLID_PASSAGE = (
    """Scientists classify matter into three primary states based on physical characteristics. In a solid,
the tiny particles are locked into place, which makes the object hard and firm. In a liquid, the
particles are close but fluid, allowing the material to flow and be poured. In a gas, the particles
have a lot of energy and fly around in all directions, spreading out as far as possible."""
)

messages = [
    {
        "role": "user",
        "content": (
            f"{SYSTEM_PROMPT}\n\n"
            f"Passage:\n{SOLID_PASSAGE}"
        )
    }
]

input_ids = tokenizer.apply_chat_template(
    messages,
    return_tensors        = "pt",
    add_generation_prompt = True
).to("cuda")

output = model.generate(
    input_ids,
    max_new_tokens = 400,
    temperature    = 0.3,
    do_sample      = True,
)

response = tokenizer.decode(
    output[0][input_ids.shape[1]:],
    skip_special_tokens = True
)

# ── Display result ───────────────────────────────────────────────
print("=" * 60)
print("MCQ: Solid State of Matter")
print("=" * 60)

try:
    parsed = json.loads(extract_json(response))
    print(f"\nQuestion: {parsed['question']}\n")
    for letter, choice in parsed["options"].items():
        marker = "  ◀ CORRECT" if letter == parsed["answer_key"] else ""
        print(f"  {letter})  {choice}{marker}")
    print(f"\nAnswer     : {parsed['answer_key']}")
    print(f"Rationale  : {parsed['rationale']}")
    print("\n✅ Valid MCQ generated successfully.")
except json.JSONDecodeError as e:
    print(f"❌ JSON parse error: {e}")
    print("Raw output:\n", response)

In [ ]:
# ── CELL 6: Install llama-cpp-python with CUDA support (fixed) ──

# Step 1: Ensure build tools are present
!apt-get update -y -q > /dev/null
!apt-get install -y cmake ninja-build libopenblas-dev 2>/dev/null | tail -3

# Step 2: Upgrade cmake in Python env (Colab's system cmake can be outdated)
!pip install cmake --upgrade -q

# Step 3: Install llama-cpp-python with the CORRECT CUDA flag for v0.2+
# DLLAMA_CUDA=on  ← old flag (v0.1.x), silently ignored on newer versions
# DGGML_CUDA=ON   ← correct flag (v0.2.x and above)
%env CMAKE_ARGS=-DGGML_CUDA=ON
%env FORCE_CMAKE=1
!pip install llama-cpp-python --no-cache-dir --force-reinstall --upgrade

# Step 4: Confirm GPU backend was compiled in (Runs in an isolated shell to bypass kernel caching)
!python -c "from llama_cpp import llama_cpp; has_gpu = hasattr(llama_cpp, 'GGML_USE_CUBLAS') or hasattr(llama_cpp, 'GGML_USE_CUDA') or 'cuda' in str(llama_cpp.__file__).lower(); print('\nGPU backend available:', has_gpu); print('llama_cpp loaded from:', llama_cpp.__file__)"

In [ ]:
# ── CELL 7: GGUF INFERENCE TEST ──────────────────────────────────
# Tests the exported .gguf file exactly as your React Native app
# would call it. This confirms the quantised model produces valid
# MCQs and that the Phi-3 chat template is respected.
#
# FIX 4: original code used the wrong path
#   WRONG: /content/phi3_science_model.gguf
#   RIGHT: /content/phi3_science_model_gguf/phi-3-mini-4k-instruct.Q4_K_M.gguf

import json, re
from llama_cpp import Llama

GGUF_PATH = "/content/phi3_science_model_gguf/phi-3-mini-4k-instruct.Q4_K_M.gguf"

print(f"Loading GGUF model from: {GGUF_PATH}")
llm = Llama(
    model_path   = GGUF_PATH,   # FIX 4: corrected path
    n_gpu_layers = -1,           # offload all layers to GPU
    n_ctx        = 2048,
    verbose      = False
)
print("Model loaded.\n")

def extract_json(text: str) -> str:
    text = re.sub(r"```(?:json)?", "", text).strip()
    match = re.search(r"\{.*\}", text, re.DOTALL)
    return match.group(0).strip() if match else text.strip()

# ── Single passage test ──────────────────────────────────────────
# Phi-3 chat format: <|user|>\n{content}<|end|>\n<|assistant|>\n
TEST_PASSAGE = chunks[0]
prompt = (
    f"<|user|>\n{SYSTEM_PROMPT}\n\nPassage:\n{TEST_PASSAGE}<|end|>\n"
    f"<|assistant|>\n"
)

result = llm(
    prompt,
    max_tokens  = 400,
    temperature = 0.3,
    stop        = ["<|end|>", "<|user|>"]
)
raw_output = result["choices"][0]["text"]

print("=" * 60)
print("GGUF RAW OUTPUT:")
print("=" * 60)
print(raw_output)
print()

# ── Validate JSON ─────────────────────────────────────────────────
REQUIRED_KEYS = {"question", "options", "answer_key", "rationale"}
print("=" * 60)
print("GGUF VALIDATION:")
print("=" * 60)
try:
    parsed = json.loads(extract_json(raw_output))
    missing = REQUIRED_KEYS - parsed.keys()
    if missing:
        print(f"⚠️  Missing keys: {missing}")
    else:
        print("✅  GGUF model produces valid MCQ JSON")
        print(f"\nQuestion   : {parsed['question']}")
        for letter, text in parsed['options'].items():
            marker = " ◀ CORRECT" if letter == parsed['answer_key'] else ""
            print(f"  {letter}) {text}{marker}")
        print(f"\nRationale  : {parsed['rationale']}")
        print("\n🟢 GGUF model is ready for React Native deployment.")
except json.JSONDecodeError as e:
    print(f"❌  JSON parse error: {e}")
    print("Attempted to parse:", extract_json(raw_output)[:300])

# ── Multi-chunk GGUF stress test (optional) ───────────────────────
print("\n" + "=" * 60)
print("GGUF STRESS TEST — 5 random chunks")
print("=" * 60)

import random
random.seed(42)
sample = random.sample(chunks, min(5, len(chunks)))

for idx, chunk in enumerate(sample):
    p = (
        f"<|user|>\n{SYSTEM_PROMPT}\n\nPassage:\n{chunk}<|end|>\n"
        f"<|assistant|>\n"
    )
    r = llm(p, max_tokens=400, temperature=0.3, stop=["<|end|>", "<|user|>"])
    out = r["choices"][0]["text"]
    try:
        parsed = json.loads(extract_json(out))
        missing = REQUIRED_KEYS - parsed.keys()
        status = "✅ PASS" if not missing else f"⚠️  WARN missing={missing}"
    except Exception as e:
        status = f"❌ FAIL ({e})"
    print(f"  Sample {idx+1}: {status}")